In [1]:
conda install scikit-learn

Channels:
 - conda-forge
 - gurobi
 - defaults
Platform: win-64
Solving environment: ...working... done

# All requested packages already installed.


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import joblib
from sklearn.model_selection import train_test_split


Make sure fights_cleaned.csv is updated by running data_cleaning.ipynb if changes are made

In [3]:
# Set paths
PROJECT_ROOT = Path('..').resolve()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

events_path = DATA_RAW / 'ufc_events.csv'
fights_path = DATA_PROCESSED / 'fights_cleaned.csv'
fighters_path = DATA_RAW / 'ufc_fighters.csv'

(events_path, fights_path)

(WindowsPath('C:/Users/maxle/Documents/ML-Project-clone/data/raw/ufc_events.csv'),
 WindowsPath('C:/Users/maxle/Documents/ML-Project-clone/data/processed/fights_cleaned.csv'))

In [4]:
# Load raw data

events = pd.read_csv(events_path)
fights = pd.read_csv(fights_path)
fighters = pd.read_csv(fighters_path)

print(events.columns)
print(fights.columns)

Index(['event_id', 'event_name', 'event_date', 'url_link'], dtype='str')
Index(['f1_id', 'f2_id', 'f1_name', 'f2_name', 'weight_class', 'f1_age_during',
       'f2_age_during', 'f1_height_cm', 'f2_height_cm', 'f1_knockdowns',
       'f2_knockdowns', 'f1_sig_strike_atts', 'f2_sig_strike_atts',
       'f1_sig_strikes', 'f2_sig_strikes', 'f1_tot_strike_atts',
       'f2_tot_strike_atts', 'f1_tot_strikes', 'f2_tot_strikes',
       'f1_takedown_atts', 'f2_takedown_atts', 'f1_takedowns', 'f2_takedowns',
       'f1_submissions', 'f2_submissions', 'f1_reversals', 'f2_reversals',
       'f1_ctrl_time', 'f2_ctrl_time', 'f1_head_strike_atts',
       'f2_head_strike_atts', 'f1_head_strikes', 'f2_head_strikes',
       'f1_body_strike_atts', 'f2_body_strike_atts', 'f1_body_strikes',
       'f2_body_strikes', 'f1_leg_strike_atts', 'f2_leg_strike_atts',
       'f1_leg_strikes', 'f2_leg_strikes', 'f1_dist_strike_atts',
       'f2_dist_strike_atts', 'f1_dist_strikes', 'f2_dist_strikes',
       'f1_clinc

In [5]:
# Join fights to events on event URL so each fight has its event date

# In events: URL column is `url_link`
# In fights: URL column is `event_url`

# This merge keeps **all** columns from both tables
fights_with_event = fights.merge(
    events,
    left_on='event_url',
    right_on='url_link',
    how='left',
)

# Just show a few columns for a quick visual check
fights_with_event[['f1_name', 'f2_name', 'event_name', 'event_date']].head()

,f1_name,f2_name,event_name,event_date
0,Cory Sandhagen,Umar Nurmagomedov,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03
1,Shara Magomedov,Michal Oleksiejczuk,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03
2,Jai Herbert,Rolando Bedoya,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03
3,Azamat Murzakanov,Alonzo Menifield,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03
4,Sedriques Dumas,Denis Tiuliulin,UFC Fight Night: Sandhagen vs. Nurmagomedov,2024-08-03


In [6]:
# Prepare joined data (in memory only; we do not save fights_with_event.csv)

DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Shared list of stat columns for cumulative stats and averages (used in later cells)
STAT_COLS = [
    'knockdowns', 'sig_strike_atts', 'sig_strikes', 'tot_strike_atts', 'tot_strikes',
    'takedown_atts', 'takedowns', 'submissions', 'reversals', 'ctrl_time',
    'head_strike_atts', 'head_strikes', 'body_strike_atts', 'body_strikes',
    'leg_strike_atts', 'leg_strikes', 'dist_strike_atts', 'dist_strikes',
    'clinch_atts', 'clinchs', 'ground_atts', 'grounds', 'total_fight_time',
]

cols_to_drop = ['url_link', 'event_url', 'fights_url']
fights_with_event_processed = fights_with_event.drop(columns=cols_to_drop, errors='ignore')
fights_with_event_processed = fights_with_event_processed.sort_values('event_date', ascending=True)

In [7]:
# Compute last-4-fight rolling averages (recency features)
# Must run BEFORE cumulative stats overwrite the raw columns

for side in ['f1', 'f2']:
    fighter_id_col = f'{side}_id'
    for metric in STAT_COLS:
        col = f'{side}_{metric}'
        if col in fights_with_event_processed.columns:
            last4_col = f'{side}_{metric}_last4'
            fights_with_event_processed[last4_col] = (
                fights_with_event_processed
                .groupby(fighter_id_col)[col]
                .transform(
                    lambda s: s.fillna(0).shift(1).rolling(window=4, min_periods=1).mean().fillna(0)
                )
            )

C:\Users\maxle\AppData\Local\Temp\ipykernel_28340\3639851891.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fights_with_event_processed[last4_col] = (


In [8]:
# Make per-fighter cumulative stats so each row shows totals *before* that fight

for side in ['f1', 'f2']:
    fighter_id_col = f'{side}_id'
    for metric in STAT_COLS:
        col = f'{side}_{metric}'
        if col in fights_with_event_processed.columns:
            fights_with_event_processed[col] = (
                fights_with_event_processed
                .groupby(fighter_id_col)[col]
                .transform(lambda s: s.fillna(0).cumsum().shift(fill_value=0))
            )

In [9]:
# Experience and win/loss/draw records are computed in the next cell (experience = wins + losses + draws).

In [10]:
# Add per-fighter prior record (wins/draws/losses before each fight)

# fights_with_event_processed is already sorted by event_date
df = fights_with_event_processed.reset_index(drop=True).reset_index().rename(columns={"index": "fight_index"})

# Long format: one row per fighter per fight
rows_f1 = df[["fight_index", "event_date", "event_id", "result", "f1_id", "f2_id"]].rename(
    columns={"f1_id": "fighter_id", "f2_id": "opponent_id"}
)
rows_f1["side"] = "f1"

rows_f2 = df[["fight_index", "event_date", "event_id", "result", "f1_id", "f2_id"]].rename(
    columns={"f2_id": "fighter_id", "f1_id": "opponent_id"}
)
rows_f2["side"] = "f2"

long = pd.concat([rows_f1, rows_f2], ignore_index=True)

# Ensure result and IDs are comparable (e.g. all numeric); avoid string vs int mismatch
long["result"] = pd.to_numeric(long["result"], errors="coerce")
long["fighter_id"] = pd.to_numeric(long["fighter_id"], errors="coerce")
long["opponent_id"] = pd.to_numeric(long["opponent_id"], errors="coerce")

# Outcome flags from the fighter's perspective
long["win_flag"] = (long["result"] == long["fighter_id"]).astype(int)
long["loss_flag"] = (long["result"] == long["opponent_id"]).astype(int)
long["draw_flag"] = (
    ~((long["result"] == long["fighter_id"]) | (long["result"] == long["opponent_id"]))
).astype(int)

# Sort by fighter and time; cumulative record *before* each fight (shift within each fighter)
long = long.sort_values(["fighter_id", "event_date", "event_id", "fight_index", "side"]).reset_index(drop=True)
# Shift must be within each fighter's group so first fight gets 0, not a global shift
long["prior_wins"] = long.groupby("fighter_id")["win_flag"].transform(lambda s: s.cumsum().shift(1, fill_value=0))
long["prior_losses"] = long.groupby("fighter_id")["loss_flag"].transform(lambda s: s.cumsum().shift(1, fill_value=0))
long["prior_draws"] = long.groupby("fighter_id")["draw_flag"].transform(lambda s: s.cumsum().shift(1, fill_value=0))

# Map back to wide table
f1_records = (
    long[long["side"] == "f1"][["fight_index", "prior_wins", "prior_losses", "prior_draws"]]
    .rename(columns={"prior_wins": "f1_wins", "prior_losses": "f1_losses", "prior_draws": "f1_draws"})
)
f2_records = (
    long[long["side"] == "f2"][["fight_index", "prior_wins", "prior_losses", "prior_draws"]]
    .rename(columns={"prior_wins": "f2_wins", "prior_losses": "f2_losses", "prior_draws": "f2_draws"})
)

fights_with_event_processed = (
    df.merge(f1_records, on="fight_index", how="left")
    .merge(f2_records, on="fight_index", how="left")
    .drop(columns=["fight_index"])
)

# Experience = total prior fights = wins + losses + draws (overwrite cumcount from earlier)
fights_with_event_processed["f1_experience"] = (
    fights_with_event_processed["f1_wins"] + fights_with_event_processed["f1_losses"] + fights_with_event_processed["f1_draws"]
)
fights_with_event_processed["f2_experience"] = (
    fights_with_event_processed["f2_wins"] + fights_with_event_processed["f2_losses"] + fights_with_event_processed["f2_draws"]
)

C:\Users\maxle\AppData\Local\Temp\ipykernel_28340\1372236789.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df = fights_with_event_processed.reset_index(drop=True).reset_index().rename(columns={"index": "fight_index"})
C:\Users\maxle\AppData\Local\Temp\ipykernel_28340\1372236789.py:55: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  fights_with_event_processed["f1_experience"] = (
C:\Users\maxle\AppData\Local\Temp\ipykernel_28340\1372236789.py:58: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

In [11]:
# Convert cumulative stats into per-fight averages (over prior fights)

for side in ['f1', 'f2']:
    exp_col = f'{side}_experience'
    exp = fights_with_event_processed[exp_col].replace(0, pd.NA)
    for metric in STAT_COLS:
        col = f'{side}_{metric}'
        if col in fights_with_event_processed.columns:
            avg_col = fights_with_event_processed[col] / exp
            fights_with_event_processed[col] = avg_col.fillna(0).astype(float)

In [12]:
# (trainable_data.csv is built in the next cell when we write the model base)

In [13]:
# Records and experience are computed in cell 6; model base and trainable are written in the next cell.

In [14]:
# Build modeling-ready CSV with selected columns

# Start from the processed, sorted, URL-free fights data
base = fights_with_event_processed.copy()

selected_cols = [
    'fight_id',
    'f1_id', 'f2_id',
    'f1_name', 'f2_name',
    'weight_class',
    'f1_age_during', 'f2_age_during',
    'f1_height_cm', 'f2_height_cm',
    'f1_knockdowns', 'f2_knockdowns',
    'f1_sig_strike_atts', 'f2_sig_strike_atts',
    'f1_sig_strikes', 'f2_sig_strikes',
    'f1_tot_strike_atts', 'f2_tot_strike_atts',
    'f1_tot_strikes', 'f2_tot_strikes',
    'f1_takedown_atts', 'f2_takedown_atts',
    'f1_takedowns', 'f2_takedowns',
    'f1_submissions', 'f2_submissions',
    'f1_reversals', 'f2_reversals',
    'f1_ctrl_time', 'f2_ctrl_time',
    'f1_head_strike_atts', 'f2_head_strike_atts',
    'f1_head_strikes', 'f2_head_strikes',
    'f1_body_strike_atts', 'f2_body_strike_atts',
    'f1_body_strikes', 'f2_body_strikes',
    'f1_leg_strike_atts', 'f2_leg_strike_atts',
    'f1_leg_strikes', 'f2_leg_strikes',
    'f1_dist_strike_atts', 'f2_dist_strike_atts',
    'f1_dist_strikes', 'f2_dist_strikes',
    'f1_clinch_atts', 'f2_clinch_atts',
    'f1_clinchs', 'f2_clinchs',
    'f1_ground_atts', 'f2_ground_atts',
    'f1_grounds', 'f2_grounds',
    'f1_total_fight_time', 'f2_total_fight_time',
    'f1_experience', 'f2_experience',
    'f1_wins', 'f1_losses', 'f1_draws', 'f2_wins', 'f2_losses', 'f2_draws',
    'result', 'winner', 'mens_class_encoded', 'womens_class_encoded', 'is_title_bout',
    'event_id', 'event_name', 'event_date',
] + [f'f1_{m}_last4' for m in STAT_COLS] + [f'f2_{m}_last4' for m in STAT_COLS]

model_df = base[selected_cols]


In [15]:
# add fighter info: dob, weight, reach, stance
fighter_cols = ['fighter_id', 'fighter_dob', 'fighter_weight_lbs', 'fighter_reach_cm',	'fighter_stance']
model_df = model_df.merge(
    fighters[fighter_cols].add_prefix('f1_').rename(columns={'f1_fighter_id': 'f1_id'}), 
    left_on='f1_id', 
    right_on='f1_id',
)
model_df = model_df.merge(
    fighters[fighter_cols].add_prefix('f2_').rename(columns={'f2_fighter_id': 'f2_id'}),
    left_on='f2_id', 
    right_on='f2_id'
)

## Post feature engineering data cleaning

In [16]:
# drop rows with missing age (this happens after calculating averages over prior fights to keep stats)
before = model_df.shape[0]
model_df.dropna(subset=['f1_age_during', 'f2_age_during'], inplace=True)
after = model_df.shape[0]
print(f"Dropping {before - after} rows with missing age during values for either fighter.")

before = len(model_df)
model_df.dropna(subset=['f1_height_cm', 'f2_height_cm'], inplace=True)
print(f'Dropped {before - len(model_df)} rows with missing height. Remaining: {len(model_df)}')

Dropping 151 rows with missing age during values for either fighter.
Dropped 2 rows with missing height. Remaining: 6747


In [17]:
# # Encode fighter stances
# # Rare stances (Open Stance, Sideways) are grouped as 'Other' : 3 — only 10 total fighters

# stance_encoding = {'Orthodox': 0, 'Southpaw': 1, 'Switch': 2, 'Open Stance': 3, 'Sideways': 3}

# for side in ['f1', 'f2']:
#     col = f'{side}_fighter_stance'
#     encoded_col = f'{side}_stance_encoded'
#     model_df[encoded_col] = model_df[col].map(stance_encoding)

# print(model_df['f1_stance_encoded'].value_counts(dropna=False))
# print(model_df['f2_stance_encoded'].value_counts(dropna=False))

# # Encode stance matchup as dummy variables for logistic regression
# stance_labels = {0: 'Orthodox', 1: 'Southpaw', 2: 'Switch', 3: 'Other'}
# model_df['stance_matchup'] = model_df['f1_stance_encoded'].map(stance_labels) + '_vs_' + model_df['f2_stance_encoded'].map(stance_labels)
# model_df = pd.get_dummies(model_df, columns=['stance_matchup'], dtype=int)

# print(f"Stance matchup columns: {[c for c in model_df.columns if c.startswith('stance_matchup')]}")

In [18]:
# NEW stance encoding: one-hot encode each fighter's stance individually
# 'Other' (includes Open Stance, Sideways) is the dropped reference category
# Produces: f1_orthodox, f1_southpaw, f1_switch, f2_orthodox, f2_southpaw, f2_switch

for side in ['f1', 'f2']:
    col = f'{side}_fighter_stance'
    normalized = model_df[col].replace({'Open Stance': 'Other', 'Sideways': 'Other'})
    model_df[f'{side}_orthodox'] = (normalized == 'Orthodox').astype(int)
    model_df[f'{side}_southpaw'] = (normalized == 'Southpaw').astype(int)
    model_df[f'{side}_switch']   = (normalized == 'Switch').astype(int)
    # 'Other' is the reference category (all three = 0)

print("Stance columns:", [c for c in model_df.columns if c.endswith(('_orthodox', '_southpaw', '_switch'))])

Stance columns: ['f1_orthodox', 'f1_southpaw', 'f1_switch', 'f2_orthodox', 'f2_southpaw', 'f2_switch']


In [19]:
# for exp in [0, 1, 2, 3]:
#     count = ((model_df['f1_experience'] == exp) | (model_df['f2_experience'] == exp)).sum()
#     print(f'Rows where f1 or f2 has {exp} experience: {count}')

In [20]:
# # drop rows where a fighter has less than 1 experience
# before = len(model_df)
# model_df = model_df[(model_df['f1_experience'] > 0) & (model_df['f2_experience'] > 0)]
# print(f"Dropped {before - len(model_df)} rows with 0 experience")

In [21]:
# drop missing fighter reach values
before = len(model_df)
((model_df['f1_fighter_reach_cm'].isna()) | (model_df['f2_fighter_reach_cm'].isna())).sum()
model_df.dropna(subset=['f1_fighter_reach_cm', 'f2_fighter_reach_cm'], inplace=True)
print(f"Dropped {before - len(model_df)} rows with missing fighter reach values")


Dropped 787 rows with missing fighter reach values


In [22]:
# drop rows with missing fighter stance
before = len(model_df)
model_df.dropna(subset=['f1_fighter_stance', 'f2_fighter_stance'], inplace=True)
print(f"Dropped {before - len(model_df)} rows with missing fighter stance values")

Dropped 2 rows with missing fighter stance values


In [23]:
# fighter_dob to datetime
model_df['f1_fighter_dob'] = pd.to_datetime(model_df['f1_fighter_dob'])
model_df['f2_fighter_dob'] = pd.to_datetime(model_df['f2_fighter_dob'])

## Make Differentials

In [24]:
diff_cols = [
    ('f1_fighter_dob', 'f2_fighter_dob'),
    ('f1_height_cm', 'f2_height_cm'),
    ('f1_knockdowns', 'f2_knockdowns'),
    ('f1_sig_strike_atts', 'f2_sig_strike_atts'),
    ('f1_sig_strikes', 'f2_sig_strikes'),
    ('f1_tot_strike_atts', 'f2_tot_strike_atts'),
    ('f1_tot_strikes', 'f2_tot_strikes'),
    ('f1_takedown_atts', 'f2_takedown_atts'),
    ('f1_takedowns', 'f2_takedowns'),
    ('f1_submissions', 'f2_submissions'),
    ('f1_reversals', 'f2_reversals'),
    ('f1_ctrl_time', 'f2_ctrl_time'),
    ('f1_head_strike_atts', 'f2_head_strike_atts'),
    ('f1_head_strikes', 'f2_head_strikes'),
    ('f1_body_strike_atts', 'f2_body_strike_atts'),
    ('f1_body_strikes', 'f2_body_strikes'),
    ('f1_leg_strike_atts', 'f2_leg_strike_atts'),
    ('f1_leg_strikes', 'f2_leg_strikes'),
    ('f1_dist_strike_atts', 'f2_dist_strike_atts'),
    ('f1_dist_strikes', 'f2_dist_strikes'),
    ('f1_clinch_atts', 'f2_clinch_atts'),
    ('f1_clinchs', 'f2_clinchs'),
    ('f1_ground_atts', 'f2_ground_atts'),
    ('f1_grounds', 'f2_grounds'),
    ('f1_total_fight_time', 'f2_total_fight_time'),
    ('f1_experience', 'f2_experience'),
    ('f1_wins', 'f2_wins'),
    ('f1_losses', 'f2_losses'),
    ('f1_fighter_weight_lbs', 'f2_fighter_weight_lbs'),
    ('f1_fighter_reach_cm', 'f2_fighter_reach_cm'),
] + [(f'f1_{m}_last4', f'f2_{m}_last4') for m in STAT_COLS]
diff_cols_list = [stat for tup in diff_cols for stat in tup]

In [25]:
for f1_col, f2_col in diff_cols:
    stat_name = f1_col[3:]
    model_df[f"diff_{stat_name}"] = model_df[f1_col].to_numpy() - model_df[f2_col].to_numpy()  

# diff_fighter_dob is a timedelta — convert to float (days)
model_df['diff_fighter_dob'] = model_df['diff_fighter_dob'].dt.days.astype(float)

In [26]:
to_drop = diff_cols_list + ['f1_fighter_stance', 'f2_fighter_stance', 'event_date', 'event_name', 'result', 'weight_class', 'f1_id', 'f2_id', 'f1_name', 'f2_name', 'event_id', 'is_title_bout', 'f1_draws', 'f2_draws']
trainable = model_df.drop(columns=to_drop)

## Make duplicates

In [27]:
def make_duplicates(df):
    dupes = df.copy()

    # Negate all differentials (f1-f2 becomes f2-f1)
    diff_feature_cols = [col for col in dupes.columns if col.startswith('diff_')]
    dupes[diff_feature_cols] = -dupes[diff_feature_cols]

    # Swap individual f1/f2 stance dummies
    for stance in ['orthodox', 'southpaw', 'switch']:
        f1_col, f2_col = f'f1_{stance}', f'f2_{stance}'
        if f1_col in dupes.columns and f2_col in dupes.columns:
            dupes[f1_col], dupes[f2_col] = dupes[f2_col].copy(), dupes[f1_col].copy()

    # # Swap stance matchup dummy values for each pair (X_vs_Y <-> Y_vs_X)
    # matchup_cols = [col for col in dupes.columns if col.startswith('stance_matchup_')]
    # processed = set()
    # for col in matchup_cols:
    #     if col in processed:
    #         continue
    #     parts = col.replace('stance_matchup_', '').split('_vs_')
    #     swapped_col = f'stance_matchup_{parts[1]}_vs_{parts[0]}'
    #     if swapped_col in dupes.columns and swapped_col != col:
    #         dupes[col], dupes[swapped_col] = dupes[swapped_col].copy(), dupes[col].copy()
    #         processed.add(col)
    #         processed.add(swapped_col)

    # Swap f1/f2 raw age features
    if 'f1_age_during' in dupes.columns and 'f2_age_during' in dupes.columns:
        dupes['f1_age_during'], dupes['f2_age_during'] = dupes['f2_age_during'].copy(), dupes['f1_age_during'].copy()

    # Flip winner
    dupes['winner'] = 1 - dupes['winner']

    # Tag original vs mirror (fight_id stays the same — links both rows to the same fight)
    df = df.copy()
    df['is_mirror'] = 0
    dupes['is_mirror'] = 1

    return pd.concat([df, dupes], ignore_index=True)

In [29]:
cols_to_scale = [col for col in trainable.columns if col.startswith('diff_')] + ['f1_age_during', 'f2_age_during']

for min_exp, label in [(1, 'exp1'), (2, 'exp2')]:
    # Filter on model_df (before trainable is built, since experience cols are dropped later)
    filtered_model_df = model_df[
        (model_df['f1_experience'] >= min_exp) &
        (model_df['f2_experience'] >= min_exp)
    ].copy()

    # Rebuild trainable from the filtered model_df
    filtered_trainable = filtered_model_df.drop(columns=to_drop, errors='ignore')

    # Split before duplicating to prevent mirror fights leaking across splits
    train, temp = train_test_split(filtered_trainable, test_size=0.30, random_state=42)
    val, test   = train_test_split(temp, test_size=2/3, random_state=42)

    # Fit scaler on training data only
    scaler = StandardScaler()
    train[cols_to_scale] = scaler.fit_transform(train[cols_to_scale])
    val[cols_to_scale]   = scaler.transform(val[cols_to_scale])
    test[cols_to_scale]  = scaler.transform(test[cols_to_scale])

    joblib.dump(scaler, DATA_PROCESSED / f'scaler_{label}.pkl')

    # Symmetrize each split independently
    train = make_duplicates(train)
    val   = make_duplicates(val)
    test  = make_duplicates(test)

    print(f"[{label}] Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

    # NOTE: when building X in model notebooks, always exclude metadata columns:
    # X = split.drop(columns=['winner', 'fight_id', 'is_mirror'])
    train.to_csv(DATA_PROCESSED / f'fights_training_{label}.csv',   index=False)
    val.to_csv(  DATA_PROCESSED / f'fights_validation_{label}.csv', index=False)
    test.to_csv( DATA_PROCESSED / f'fights_testing_{label}.csv',    index=False)

[exp1] Train: 6644 | Val: 950 | Test: 1900
[exp2] Train: 5212 | Val: 744 | Test: 1492
